# Mailroom dataset browser

Docile-style browser ([rossumai/docile `tools/dataset_browser.ipynb`](https://github.com/rossumai/docile/blob/main/docile/tools/dataset_browser.ipynb) pattern):
a **thin notebook** over a reusable tool module (`dataset_browser.py`, same folder).

It browses the pilot sample set defined by `docs/examples/samples/manifest.csv`
(30 rows) and — when the pipeline has run — joins each sample with its observed
record from the catalog (`data/mailroom.db`, opened **read-only**).
Published Hugging Face corpora (CUAD, LegalBench, Enron, DE-SynPUF) are
notebook `11_huggingface_corpora` — offline Dataset Viewer snapshot, optional live Hub refresh:

| layer | source | meaning |
|---|---|---|
| ground truth | manifest CSV | expected doc class / stage / fields, provenance (CUAD & external = REAL committed legal documents; everything else = synthetic) |
| observed | catalog SQLite | stage actually reached, confidences, extracted payload, model/prompt/cost provenance |

No network, no LLM calls.

In [ ]:
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "docs" / "examples" / "samples" / "manifest.csv").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT))

from notebooks.dataset_browser import (
    DatasetBrowser,
    join_catalog,
    load_catalog_state,
    load_manifest,
    summarize,
)
print("repo root:", ROOT)

## Load the dataset

Manifest rows materialize to `data/samples/` — if PDFs are missing, run
`PYTHONPATH=src python src/scripts/prepare_samples.py` first.

In [ ]:
records = load_manifest()
joined = join_catalog(records, load_catalog_state())
summarize(joined)

## Display the dataset

Interactive picker (needs the `notebooks` extra: `pip install -e ".[notebooks]"`);
falls back to a plain-text listing otherwise.

In [ ]:
browser = DatasetBrowser(records)
# pick a sample:  browser.show("contract_01")

## Inspect one sample in detail

In [ ]:
browser.show("contract_01")